## 12. Usage Examples

You can now use the loaded data and functions to explore the StatsBomb dataset. Here are some examples:

### Example Queries:
```python
# Get all matches for a team
barcelona_matches = get_matches_for_team('Barcelona')

# Get matches from a specific competition
premier_league = get_matches_by_competition('Premier League')

# Analyze a specific match
match_analysis = analyze_match_events(15946)

# Get all events for a specific player in a match
player_events = get_player_events(15946, 'Lionel Andrés Messi Cuccittini')

# Filter competitions by criteria
recent_competitions = df_competitions[df_competitions['season_name'] >= '2020/2021']

# Complex event filtering
goals = df_sample_events[
    (df_sample_events['type.name'] == 'Shot') & 
    (df_sample_events['shot.outcome.name'] == 'Goal')
]
```

### Data Exploration Tips:
- Use `df.info()` to see column types and non-null counts
- Use `df.describe()` for numerical statistics
- Use `df.head()`, `df.tail()` to preview data
- Use `df.columns` to see all available columns
- Use `df['column'].value_counts()` to see frequency distributions

In [17]:
print("StatsBomb Data Summary")
print("=" * 60)
print(f"Total Competitions/Seasons: {len(df_competitions)}")
print(f"Total Matches: {len(df_matches)}")
print(f"Total Lineup Files: {len(list(LINEUPS_DIR.glob('*.json')))}")
print(f"Total Event Files: {len(list(EVENTS_DIR.glob('*.json')))}")

print("\n\nCompetitions Breakdown:")
print("-" * 60)
comp_breakdown = df_competitions.groupby('competition_gender')['competition_name'].nunique()
print(f"Male competitions: {comp_breakdown.get('male', 0)}")
print(f"Female competitions: {comp_breakdown.get('female', 0)}")

print("\n\nData Availability:")
print("-" * 60)
if 'match_available_360' in df_competitions.columns:
    matches_with_360 = df_competitions[df_competitions['match_available_360'].notna()]
    print(f"Competitions with 360 data: {len(matches_with_360)}")

print("\n\nMost Recent Updates:")
print("-" * 60)
if 'match_updated' in df_competitions.columns:
    df_competitions['match_updated'] = pd.to_datetime(df_competitions['match_updated'])
    recent = df_competitions.nlargest(5, 'match_updated')[['competition_name', 'season_name', 'match_updated']]
    print(recent.to_string(index=False))

StatsBomb Data Summary
Total Competitions/Seasons: 75
Total Matches: 3464
Total Lineup Files: 3464
Total Event Files: 3464


Competitions Breakdown:
------------------------------------------------------------
Male competitions: 17
Female competitions: 4


Data Availability:
------------------------------------------------------------
Competitions with 360 data: 11


Most Recent Updates:
------------------------------------------------------------
 competition_name season_name              match_updated
UEFA Women's Euro        2025 2025-07-28 14:19:20.467348
          Serie A   2015/2016 2025-07-21 05:01:00.434081
Women's World Cup        2023 2025-07-14 10:07:06.620906
          La Liga   2017/2018 2025-07-14 10:01:16.674864
          La Liga   2005/2006 2025-06-24 14:05:07.849675


## 11. Summary Statistics

In [16]:
def get_matches_for_team(team_name, df=df_matches):
    """Get all matches for a specific team."""
    return df[
        (df['home_team.home_team_name'] == team_name) | 
        (df['away_team.away_team_name'] == team_name)
    ]

def get_matches_by_competition(competition_name, df=df_matches):
    """Get all matches for a specific competition."""
    if 'competition.competition_name' in df.columns:
        return df[df['competition.competition_name'] == competition_name]
    return pd.DataFrame()

def analyze_match_events(match_id):
    """Analyze events for a specific match."""
    events = load_events(match_id)
    if events is None:
        return None
    
    analysis = {
        'total_events': len(events),
        'event_types': events['type.name'].value_counts().to_dict(),
        'periods': events['period'].unique().tolist(),
        'teams': events['team.name'].unique().tolist() if 'team.name' in events.columns else []
    }
    
    return analysis

def get_player_events(match_id, player_name):
    """Get all events for a specific player in a match."""
    events = load_events(match_id)
    if events is None or 'player.name' not in events.columns:
        return pd.DataFrame()
    
    return events[events['player.name'] == player_name]

print("✓ Advanced query functions loaded")
print("\nAvailable functions:")
print("  - get_matches_for_team(team_name)")
print("  - get_matches_by_competition(competition_name)")
print("  - analyze_match_events(match_id)")
print("  - get_player_events(match_id, player_name)")

✓ Advanced query functions loaded

Available functions:
  - get_matches_for_team(team_name)
  - get_matches_by_competition(competition_name)
  - analyze_match_events(match_id)
  - get_player_events(match_id, player_name)


## 10. Advanced Query Functions

In [15]:
if df_sample_events is not None:
    print("Example 1: Filter events by type (e.g., Goals)")
    print("=" * 60)
    shots = df_sample_events[df_sample_events['type.name'] == 'Shot']
    print(f"Total shots in match: {len(shots)}")
    if 'shot.outcome.name' in df_sample_events.columns:
        print("\nShot outcomes:")
        print(shots['shot.outcome.name'].value_counts())
    
    print("\n\nExample 2: Filter events by player")
    print("=" * 60)
    if 'player.name' in df_sample_events.columns:
        player_events = df_sample_events['player.name'].value_counts().head(10)
        print("Top 10 most active players:")
        print(player_events)
    
    print("\n\nExample 3: Filter events by time period")
    print("=" * 60)
    first_half = df_sample_events[df_sample_events['period'] == 1]
    second_half = df_sample_events[df_sample_events['period'] == 2]
    print(f"First half events: {len(first_half)}")
    print(f"Second half events: {len(second_half)}")
    
    print("\n\nExample 4: Filter passes by team")
    print("=" * 60)
    passes = df_sample_events[df_sample_events['type.name'] == 'Pass']
    print(f"Total passes: {len(passes)}")
    if 'team.name' in passes.columns:
        print("\nPasses by team:")
        print(passes['team.name'].value_counts())

Example 1: Filter events by type (e.g., Goals)
Total shots in match: 31

Shot outcomes:
shot.outcome.name
Off T      15
Blocked     8
Goal        3
Saved       3
Wayward     2
Name: count, dtype: int64


Example 2: Filter events by player
Top 10 most active players:
player.name
Diego Armando Maradona       230
Antônio de Oliveira Filho    190
Ricardo Rogério de Brito     189
Fernando De Napoli           185
Alessandro Renica            176
Srečko Katanec               152
Ásgeir Sigurvinsson          150
Karl Allgöwer                137
Maurizio Gaudino             127
Giovanni Francini            122
Name: count, dtype: int64


Example 3: Filter events by time period
First half events: 1537
Second half events: 1353


Example 4: Filter passes by team
Total passes: 773

Passes by team:
team.name
Napoli           477
VfB Stuttgart    296
Name: count, dtype: int64


## 9. Query Examples - Events

In [14]:
# Example 1: Filter matches by team
if 'home_team.home_team_name' in df_matches.columns:
    print("Example 1: Find all matches for a specific team")
    print("=" * 60)
    # Get unique team names (first 10)
    all_teams = set(df_matches['home_team.home_team_name'].dropna().unique()) | set(df_matches['away_team.away_team_name'].dropna().unique())
    print(f"Sample teams available: {list(all_teams)[:10]}")
    
    # Filter for a specific team (use first available team)
    if len(all_teams) > 0:
        team_name = list(all_teams)[0]
        team_matches = df_matches[
            (df_matches['home_team.home_team_name'] == team_name) | 
            (df_matches['away_team.away_team_name'] == team_name)
        ]
        print(f"\n{team_name} matches: {len(team_matches)}")
        if len(team_matches) > 0:
            print(team_matches[['match_date', 'home_team.home_team_name', 'away_team.away_team_name', 'home_score', 'away_score']].head())

print("\n\nExample 2: Filter by date range")
print("=" * 60)
df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])
recent_matches = df_matches[df_matches['match_date'] >= '2020-01-01']
print(f"Matches from 2020 onwards: {len(recent_matches)}")

print("\n\nExample 3: Filter by competition and season")
print("=" * 60)
if 'competition.competition_name' in df_matches.columns and 'season.season_name' in df_matches.columns:
    comp_season = df_matches.groupby(['competition.competition_name', 'season.season_name']).size().reset_index(name='match_count')
    print(comp_season.head(10))

Example 1: Find all matches for a specific team
Sample teams available: ['Sassuolo', 'Mumbai City', 'Iceland', 'Boca Juniors', 'Borussia Dortmund', 'Georgia', 'Sevilla', 'Portsmouth', 'Poland', 'Jamshedpur']

Sassuolo matches: 38
      match_date home_team.home_team_name away_team.away_team_name  \
2638  2016-01-24                 Sassuolo                  Bologna   
2646  2015-12-20            Hellas Verona                 Sassuolo   
2652  2015-10-25                 AC Milan                 Sassuolo   
2678  2016-03-11                 Juventus                 Sassuolo   
2681  2016-03-06                 Sassuolo                 AC Milan   

      home_score  away_score  
2638           0           2  
2646           1           1  
2652           2           1  
2678           1           0  
2681           2           0  


Example 2: Filter by date range
Matches from 2020 onwards: 806


Example 3: Filter by competition and season
  competition.competition_name season.season_name  m

## 8. Query Examples - Matches

In [13]:
# Example 1: Filter competitions by country
print("Example 1: Competitions in England")
print("=" * 60)
england_comps = df_competitions[df_competitions['country_name'] == 'England']
print(england_comps[['competition_name', 'season_name', 'competition_gender']])

print("\n\nExample 2: Filter by competition name")
print("=" * 60)
# Find all Champions League seasons
if 'Champions League' in df_competitions['competition_name'].values:
    champions_league = df_competitions[df_competitions['competition_name'] == 'Champions League']
    print(f"Champions League seasons available: {len(champions_league)}")
    print(champions_league[['season_name', 'match_available']])

print("\n\nExample 3: Filter by gender")
print("=" * 60)
female_comps = df_competitions[df_competitions['competition_gender'] == 'female']
print(f"Female competitions available: {len(female_comps)}")
print(female_comps[['competition_name', 'country_name', 'season_name']].head())

Example 1: Competitions in England
           competition_name season_name competition_gender
25  FA Women's Super League   2020/2021             female
26  FA Women's Super League   2019/2020             female
27  FA Women's Super League   2018/2019             female
64           Premier League   2015/2016               male
65           Premier League   2003/2004               male


Example 2: Filter by competition name
Champions League seasons available: 18
   season_name             match_available
3    2018/2019  2025-05-08T15:10:50.835274
4    2017/2018  2024-02-13T02:35:28.134882
5    2016/2017  2024-02-13T02:37:32.205154
6    2015/2016  2024-06-12T07:45:38.786894
7    2014/2015  2024-02-12T12:49:54.914228
8    2013/2014  2024-02-12T12:48:48.479157
9    2012/2013  2024-02-12T12:47:34.340413
10   2011/2012  2024-02-13T02:36:35.698340
11   2010/2011  2024-02-12T12:53:03.944320
12   2009/2010  2024-02-12T12:49:25.017694
13   2008/2009  2024-02-13T07:02:54.657056
14   2006/2007  

## 7. Query Examples - Competitions

In [12]:
if df_sample_events is not None:
    print("Event Types in Sample Match:")
    print("=" * 60)
    event_counts = df_sample_events['type.name'].value_counts()
    print(event_counts)
    
    print(f"\n\nTotal unique event types: {df_sample_events['type.name'].nunique()}")

Event Types in Sample Match:
type.name
Pass               773
Ball Receipt*      679
Carry              539
Pressure           262
Ball Recovery      121
Duel                97
Clearance           74
Foul Committed      50
Foul Won            46
Miscontrol          42
Block               36
Goal Keeper         34
Shot                31
Dribble             31
Interception        24
Dispossessed        18
Dribbled Past       14
Half Start           4
Half End             4
Tactical Shift       2
Substitution         2
Starting XI          2
Injury Stoppage      2
Player On            1
Player Off           1
Bad Behaviour        1
Name: count, dtype: int64


Total unique event types: 26


### Explore Event Types

In [11]:
def load_events(match_id):
    """Load event data for a specific match."""
    event_file = EVENTS_DIR / f"{match_id}.json"
    
    if not event_file.exists():
        print(f"Event file not found for match {match_id}")
        return None
    
    with open(event_file, 'r') as f:
        events_data = json.load(f)
    
    return pd.json_normalize(events_data)

# Load a sample event data
if len(df_matches) > 0:
    sample_match_id = df_matches['match_id'].iloc[0]
    df_sample_events = load_events(sample_match_id)
    
    if df_sample_events is not None:
        print(f"Sample events for match {sample_match_id}:")
        print("=" * 60)
        print(f"Total events: {len(df_sample_events)}")
        print(f"Columns: {df_sample_events.shape[1]}")
        print(f"\nFirst few events:")
        print(df_sample_events[['minute', 'second', 'type.name', 'team.name', 'player.name']].head(10))
else:
    print("No matches available")

Sample events for match 3887188:
Total events: 2890
Columns: 116

First few events:
   minute  second      type.name      team.name          player.name
0       0       0    Starting XI         Napoli                  NaN
1       0       0    Starting XI  VfB Stuttgart                  NaN
2       0       0     Half Start         Napoli                  NaN
3       0       0     Half Start  VfB Stuttgart                  NaN
4       0       1           Pass  VfB Stuttgart         Fritz Walter
5       0       2  Ball Receipt*  VfB Stuttgart     Maurizio Gaudino
6       0       2           Pass  VfB Stuttgart     Maurizio Gaudino
7       0       4  Ball Receipt*  VfB Stuttgart  Ásgeir Sigurvinsson
8       0       4          Carry  VfB Stuttgart  Ásgeir Sigurvinsson
9       0       5           Pass  VfB Stuttgart  Ásgeir Sigurvinsson


## 6. Load Events Data (Sample)

In [10]:
def load_lineup(match_id):
    """Load lineup data for a specific match."""
    lineup_file = LINEUPS_DIR / f"{match_id}.json"
    
    if not lineup_file.exists():
        print(f"Lineup file not found for match {match_id}")
        return None
    
    with open(lineup_file, 'r') as f:
        lineup_data = json.load(f)
    
    # Process lineup data
    all_players = []
    for team in lineup_data:
        team_id = team['team_id']
        team_name = team['team_name']
        for player in team['lineup']:
            player_info = {
                'team_id': team_id,
                'team_name': team_name,
                'player_id': player['player_id'],
                'player_name': player['player_name'],
                'player_nickname': player.get('player_nickname'),
                'jersey_number': player['jersey_number']
            }
            all_players.append(player_info)
    
    return pd.DataFrame(all_players)

# Load a sample lineup (using first match_id from matches)
if len(df_matches) > 0:
    sample_match_id = df_matches['match_id'].iloc[0]
    df_sample_lineup = load_lineup(sample_match_id)
    
    if df_sample_lineup is not None:
        print(f"Sample lineup for match {sample_match_id}:")
        print("=" * 60)
        print(df_sample_lineup)
        print(f"\nTotal players: {len(df_sample_lineup)}")
else:
    print("No matches available")

Sample lineup for match 3887188:
    team_id      team_name  player_id                player_name  \
0       227         Napoli      19624               Ciro Ferrara   
1       227         Napoli      38641     Diego Armando Maradona   
2       227         Napoli      39815          Giuliano Giuliani   
3       227         Napoli      39816        Giancarlo Corradini   
4       227         Napoli      39817          Alessandro Renica   
5       227         Napoli      39818         Antonio Carannante   
6       227         Napoli      39819          Giovanni Francini   
7       227         Napoli      39820             Massimo Crippa   
8       227         Napoli      39822           Andrea Carnevale   
9       227         Napoli      39826  Antônio de Oliveira Filho   
10      227         Napoli      39829   Ricardo Rogério de Brito   
11      227         Napoli      39843                  Luca Fusi   
12      227         Napoli      39846         Fernando De Napoli   
13      227    

## 5. Load Lineups Data (Sample)

In [9]:
# Display basic information about matches
print("Matches Data Info:")
print("=" * 60)
print(f"Total matches: {len(df_matches)}")
print(f"Columns: {df_matches.shape[1]}")
print(f"\nKey columns:")
key_cols = [col for col in df_matches.columns if not col.startswith('home_team.managers') and not col.startswith('away_team.managers')][:20]
print(key_cols)

print(f"\nDate range: {df_matches['match_date'].min()} to {df_matches['match_date'].max()}")
print(f"\nMatches by competition:")
if 'competition.competition_name' in df_matches.columns:
    print(df_matches['competition.competition_name'].value_counts())

Matches Data Info:
Total matches: 3464
Columns: 42

Key columns:
['match_id', 'match_date', 'kick_off', 'home_score', 'away_score', 'match_status', 'match_status_360', 'last_updated', 'last_updated_360', 'match_week', 'competition.competition_id', 'competition.country_name', 'competition.competition_name', 'season.season_id', 'season.season_name', 'home_team.home_team_id', 'home_team.home_team_name', 'home_team.home_team_gender', 'home_team.home_team_group', 'home_team.country.id']

Date range: 1958-06-24 to 2025-07-27

Matches by competition:
competition.competition_name
La Liga                    868
Ligue 1                    435
Premier League             418
Serie A                    381
1. Bundesliga              340
FA Women's Super League    326
FIFA World Cup             147
Women's World Cup          116
Indian Super league        115
UEFA Euro                  102
UEFA Women's Euro           62
African Cup of Nations      52
NWSL                        36
Copa America      

### Explore Matches Data

In [8]:
def load_all_matches():
    """Load all match data from all competition/season folders."""
    all_matches = []
    
    # Iterate through competition directories
    for comp_dir in MATCHES_DIR.iterdir():
        if comp_dir.is_dir():
            # Iterate through season files in each competition
            for season_file in comp_dir.glob('*.json'):
                with open(season_file, 'r') as f:
                    matches = json.load(f)
                    all_matches.extend(matches)
    
    return pd.json_normalize(all_matches)

# Load all matches
print("Loading all matches data...")
df_matches = load_all_matches()

print(f"✓ Total matches loaded: {len(df_matches)}")
print(f"DataFrame shape: {df_matches.shape}")
print(f"\nSample data:")
print(df_matches.head())

Loading all matches data...
✓ Total matches loaded: 3464
DataFrame shape: (3464, 42)

Sample data:
   match_id  match_date      kick_off  home_score  away_score match_status  \
0   3887188  1989-05-03  21:30:00.000           2           1    available   
1   3750244  1989-04-05          None           2           0    available   
2   3750245  1989-03-15  20:30:00.000           3           0    available   
3   3827767  2022-03-20  16:00:00.000           1           1    available   
4   3827335  2022-03-15  16:00:00.000           1           1    available   

  match_status_360                last_updated         last_updated_360  \
0      unscheduled  2023-06-18T19:28:39.443883                     None   
1        scheduled            2020-07-29T05:00  2021-06-13T16:17:31.694   
2        scheduled  2023-05-22T16:05:43.468317  2021-06-13T16:17:31.694   
3      unscheduled  2022-04-14T17:33:06.627987                     None   
4      unscheduled  2022-05-17T22:00:48.247246           

## 4. Load Matches Data

In [7]:
# View all unique competitions
print("Available Competitions:")
print("=" * 60)
competitions_summary = df_competitions.groupby(['competition_name', 'country_name']).size().reset_index(name='seasons_count')
print(competitions_summary.to_string(index=False))

print(f"\n\nTotal unique competitions: {df_competitions['competition_name'].nunique()}")
print(f"Total unique countries: {df_competitions['country_name'].nunique()}")

Available Competitions:
       competition_name              country_name  seasons_count
          1. Bundesliga                   Germany              2
 African Cup of Nations                    Africa              1
       Champions League                    Europe             18
           Copa America             South America              1
           Copa del Rey                     Spain              3
FA Women's Super League                   England              3
     FIFA U20 World Cup             International              1
         FIFA World Cup             International              8
    Indian Super league                     India              1
                La Liga                     Spain             18
       Liga Profesional                 Argentina              2
                Ligue 1                    France              3
    Major League Soccer  United States of America              1
                   NWSL  United States of America              1
 

## 1. Import Required Libraries

### Explore Competitions

In [6]:
# Load competitions data
with open(COMPETITIONS_FILE, 'r') as f:
    competitions_data = json.load(f)

# Convert to DataFrame
df_competitions = pd.json_normalize(competitions_data)

print(f"Total competitions/seasons available: {len(df_competitions)}")
print(f"\nDataFrame shape: {df_competitions.shape}")
print(f"\nColumns: {list(df_competitions.columns)}")
print(f"\n{df_competitions.head()}")

Total competitions/seasons available: 75

DataFrame shape: (75, 12)

Columns: ['competition_id', 'season_id', 'country_name', 'competition_name', 'competition_gender', 'competition_youth', 'competition_international', 'season_name', 'match_updated', 'match_updated_360', 'match_available_360', 'match_available']

   competition_id  season_id country_name        competition_name  \
0               9        281      Germany           1. Bundesliga   
1               9         27      Germany           1. Bundesliga   
2            1267        107       Africa  African Cup of Nations   
3              16          4       Europe        Champions League   
4              16          1       Europe        Champions League   

  competition_gender  competition_youth  competition_international  \
0               male              False                      False   
1               male              False                      False   
2               male              False                      

## 3. Load Competitions Data

In [5]:
# Define base paths
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'

# Define specific data paths
COMPETITIONS_FILE = DATA_DIR / 'competitions.json'
MATCHES_DIR = DATA_DIR / 'matches'
LINEUPS_DIR = DATA_DIR / 'lineups'
EVENTS_DIR = DATA_DIR / 'events'
THREE60_DIR = DATA_DIR / 'three-sixty'

# Verify paths exist
print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"\nDirectory Status:")
print(f"✓ Competitions file exists: {COMPETITIONS_FILE.exists()}")
print(f"✓ Matches directory exists: {MATCHES_DIR.exists()}")
print(f"✓ Lineups directory exists: {LINEUPS_DIR.exists()}")
print(f"✓ Events directory exists: {EVENTS_DIR.exists()}")
print(f"✓ Three-sixty directory exists: {THREE60_DIR.exists()}")

Base Directory: /Users/jonophillip/Statsbomb/open-data
Data Directory: /Users/jonophillip/Statsbomb/open-data/data

Directory Status:
✓ Competitions file exists: True
✓ Matches directory exists: True
✓ Lineups directory exists: True
✓ Events directory exists: True
✓ Three-sixty directory exists: True


## 2. Set Up Data Paths

In [3]:
import pandas as pd
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set pandas display options for better viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


# StatsBomb Open Data Explorer

This notebook provides tools to load, explore, and query StatsBomb soccer data using pandas.

## Data Structure:
- **competitions.json**: Information about available competitions and seasons
- **matches/**: Match data organized by competition_id/season_id
- **lineups/**: Player lineups for each match
- **events/**: Detailed event data for each match
- **three-sixty/**: 360-degree tracking data (if available)